**Librerías**

In [3]:
from totalsegmentator.python_api import totalsegmentator
import SimpleITK as sitk
import numpy as np
import os 
import cv2
from pydicom import dcmread
#from pydicom.errors import InvalidDicomError
import nibabel as nib

**Directorio**

In [4]:
"""
En esta secciuon es necesario automatizar la dirección de la ruta del archivo DICOM,
para ello se deben recibir los archivos del form upload del html.
"""

input_path = "D:/Documentos/TEC/Reto/CT_dicoms/ADX/Daniele_Morosetti_ADX_ITA_PHILIPS_2/"
output_file = "volumen.stl"
output_path = "/"

**Extracción de máscara de la aorta**

In [5]:
"""
Considerar la ubicación para guardar la máscara 
"""
#output_path = f"{input_path.split("/")[-1]}/"
output_path = "/"


print("Iniciando la segmentación de la aorta...")

totalsegmentator(input=input_path, output=output_path, roi_subset=["aorta"], fast=False, device="cpu" )
print("Segmentación completada")

Iniciando la segmentación de la aorta...

If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024



c:\Users\ferna\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Generating rough segmentation for cropping...
Converting dicom to nifti...
  found image with shape (512, 512, 388)
Resampling...
  Resampled in 7.26s
Predicting...


100%|██████████| 8/8 [00:11<00:00,  1.38s/it]


  Predicted in 33.12s
Resampling...
Converting dicom to nifti...
  found image with shape (512, 512, 388)
  cropping from (512, 512, 388) to (135, 236, 325)
Resampling...
  Resampled in 1.37s
Predicting part 1 of 1 ...


100%|██████████| 2/2 [00:22<00:00, 11.34s/it]


  Predicted in 45.69s
Resampling...
Saving segmentations...
  Saved in 21.54s
Segmentación completada


**Cortar caras de entrada y salida**

In [6]:
mask_path = f"{output_path}aorta.nii.gz"
mask = sitk.ReadImage(mask_path)
mask_limpia_3d = sitk.GetArrayFromImage(mask)

mask_limpia_3d = np.copy(mask_limpia_3d) 
total_slices = mask_limpia_3d.shape[0]

# --- VARIABLES CONFIGURABLES ---
slices_iniciales_a_borrar = 10     # Slices a borrar de la salida
slices_superiores_a_modificar = 10 # Slices a modificar de la entarda

# 1. Borrar las slices de la salida
z_con_mascara = np.where(np.any(mask_limpia_3d, axis=(1, 2)))[0]

indices_iniciales_a_borrar = z_con_mascara[:slices_iniciales_a_borrar]

for z in indices_iniciales_a_borrar:
    mask_limpia_3d[z, :, :] = 0

# 2. Modificar slices de la entrada
umbral_55 = int(total_slices * 0.55)
slices_modificadas = 0

for z in range(umbral_55, total_slices):
    
    if slices_modificadas >= slices_superiores_a_modificar:
        break 
        
    slice_actual = mask_limpia_3d[z, :, :].astype(np.uint8)
    
    if np.any(slice_actual):
        

        num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(slice_actual, connectivity=8)
        
        if num_labels >= 3:
            min_y = float('inf')
            label_a_eliminar = -1
            
            # Buscar el cuerpo más próximo al cero en el eje Y
            for i in range(1, num_labels):
                centroid_y = centroids[i][1] # El índice 1 corresponde al eje Y
                
                if centroid_y < min_y:
                    min_y = centroid_y
                    label_a_eliminar = i
            
            # Si encontramos el cuerpo, lo rellenamos con ceros
            if label_a_eliminar != -1:
                # Todo lo que corresponda a esa etiqueta se vuelve 0
                slice_actual[labels == label_a_eliminar] = 0
                
                # Actualizamos la matriz 3D
                mask_limpia_3d[z, :, :] = slice_actual
                
                # Aumentamos el contador de slices modificadas
                slices_modificadas += 1

# Guardar el resultado
mask_limpia_sitk = sitk.GetImageFromArray(mask_limpia_3d)
mask_limpia_sitk.CopyInformation(mask)


**Cortar caras y obtener las coordenadas**

In [7]:
mask_limpia_3d = sitk.GetArrayFromImage(mask_limpia_sitk)
mask_limpia_3d = np.copy(mask_limpia_3d) 
total_slices = mask_limpia_3d.shape[0]

# =========================================================================
# PARÁMETROS CONFIGURABLES
# =========================================================================
slices_offset_iliacas = 10
slices_superiores_a_modificar = 10 
umbral_limpieza_arco = 0.60         

# NUEVO: Píxeles mínimos para que un cuerpo se considere una rama aórtica (ascendente/descendente)
# y no ruido flotante. (Ajustable, 80 suele ser seguro para ramas principales)
area_minima_anatomica = 80        
# =========================================================================

# FASE 0: Cálculo de Umbrales sobre la Anatomía Real
z_activos = np.where(np.any(mask_limpia_3d, axis=(1, 2)))[0]

if len(z_activos) > 0:
    z_min = z_activos[0]   
    z_max = z_activos[-1]  
    altura_real = z_max - z_min
    
    umbral_55 = z_min + int(altura_real * 0.55)
    umbral_arco = z_min + int(altura_real * umbral_limpieza_arco)
else:
    z_min, z_max, umbral_55, umbral_arco = 0, total_slices, 0, 0

def obtener_centroide_superior(slice_data):
    img = slice_data.astype(np.uint8)
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(img, connectivity=8)
    if num_labels >= 2: 
        min_y = float('inf')
        idx_superior = -1
        for i in range(1, num_labels):
            if centroids[i][1] < min_y:
                min_y = centroids[i][1]
                idx_superior = i
        return centroids[idx_superior] 
    return None

# =========================================================================
# FASE 1: Limpieza de Ilíacas (Barrido Inverso)
# =========================================================================
z_corte_final_inferior = z_min 
for z in range(umbral_55, z_min - 1, -1):
    slice_actual = mask_limpia_3d[z, :, :].astype(np.uint8)
    if np.any(slice_actual):
        num_labels, _, _, _ = cv2.connectedComponentsWithStats(slice_actual, connectivity=8)
        if num_labels >= 3:
            z_corte_final_inferior = z + slices_offset_iliacas
            break

if z_corte_final_inferior > total_slices: z_corte_final_inferior = total_slices
mask_limpia_3d[0 : z_corte_final_inferior + 1, :, :] = 0
z_inicio_aorta = z_corte_final_inferior + 1

# =========================================================================
# FASE 2: Artefacto del 55% 
# =========================================================================
slices_modificadas = 0
ultimo_z_superior = -1

for z in range(umbral_55, total_slices):
    if slices_modificadas >= slices_superiores_a_modificar:
        break 
    slice_actual = mask_limpia_3d[z, :, :].astype(np.uint8)
    if np.any(slice_actual):
        num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(slice_actual, connectivity=8)
        if num_labels >= 3:
            min_y = float('inf')
            label_a_eliminar = -1
            for i in range(1, num_labels):
                if centroids[i][1] < min_y:
                    min_y = centroids[i][1]
                    label_a_eliminar = i
            if label_a_eliminar != -1:
                slice_actual[labels == label_a_eliminar] = 0
                mask_limpia_3d[z, :, :] = slice_actual
                slices_modificadas += 1
                ultimo_z_superior = z 

# =========================================================================
# FASE 3: Limpieza Topológica del Arco (60%+)
# =========================================================================
ultimo_z_con_aorta = umbral_arco 

for z in range(umbral_arco, total_slices):
    slice_actual = mask_limpia_3d[z, :, :].astype(np.uint8)
    
    if np.any(slice_actual):
        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(slice_actual, connectivity=8)
        
        # Si detectamos ruido o ramas adicionales (Fondo + 2 o más cuerpos)
        if num_labels > 2:
            areas = stats[1:, cv2.CC_STAT_AREA]
            # Ordenamos los índices de mayor a menor área (+1 para omitir el fondo)
            indices_ordenados = np.argsort(areas)[::-1] + 1
            
            # Siempre conservamos el cuerpo de mayor tamaño
            labels_a_conservar = [indices_ordenados[0]]
            
            # Evaluamos el segundo cuerpo más grande (la posible aorta ascendente)
            if len(indices_ordenados) >= 2:
                area_segundo = stats[indices_ordenados[1], cv2.CC_STAT_AREA]
                # Si supera el umbral, es anatomía real, lo conservamos. Si no, es ruido.
                if area_segundo > area_minima_anatomica:
                    labels_a_conservar.append(indices_ordenados[1])
            
            # Limpiamos todo lo que no esté en la lista de conservación
            mascara_conservar = np.isin(labels, labels_a_conservar)
            slice_actual[~mascara_conservar & (labels != 0)] = 0
            mask_limpia_3d[z, :, :] = slice_actual

        # Validación para la guillotina final
        if np.any(slice_actual):
            num_labels_post, _, stats_post, _ = cv2.connectedComponentsWithStats(slice_actual, connectivity=8)
            area_maxima = np.max(stats_post[1:, cv2.CC_STAT_AREA]) if num_labels_post > 1 else 0
            
            # Si aún queda estructura anatómica real, actualizamos el límite superior
            if area_maxima > area_minima_anatomica:
                ultimo_z_con_aorta = z

# Guillotina Final
if ultimo_z_con_aorta < total_slices - 1:
    mask_limpia_3d[ultimo_z_con_aorta + 1:, :, :] = 0

# =========================================================================
# FASE 4: Coordenadas y Guardado
# =========================================================================

Z_centroide = int((umbral_55 - z_min)/2)
coord = "No disponible"
if z_inicio_aorta < total_slices and np.any(mask_limpia_3d[Z_centroide]):
    cent = obtener_centroide_superior(mask_limpia_3d[Z_centroide])
    if cent is not None:
        coord = [cent[0], cent[1], Z_centroide]


# Guardado
mask_limpia_sitk = sitk.GetImageFromArray(mask_limpia_3d)
mask_limpia_sitk.CopyInformation(mask)
sitk.WriteImage(mask_limpia_sitk, 'mascara_optimizada.nii.gz')



print(f"Proceso completado. Máscara optimizada y coordenadas guardadas.")

Proceso completado. Máscara optimizada y coordenadas guardadas.


**Obtener medidas reales**

In [9]:
# 2. Read all DICOM files from the folder

""""
dicom_slices = []
for filename in os.listdir(input_path):
    filepath = os.path.join(input_path, filename)
    
    # Skip subdirectories if any exist
    if os.path.isdir(filepath):
        continue
        
    try:
        # Read the file
        dataset = dcmread(filepath)
        
        # Make sure the file actually contains image data before adding it
        if hasattr(dataset, 'pixel_array'):
            dicom_slices.append(dataset)
    except InvalidDicomError:
        # Skip files that aren't valid DICOMs (like hidden OS files, .txt files, etc.)
        continue




def get_voxel_size(slices, target_resolution=512):

    ds = slices[0]
    
    # 1. Cálculo de X e Y (ajustado por redimensión)
    original_spacing_x = float(ds.PixelSpacing[0])
    original_spacing_y = float(ds.PixelSpacing[1])
    
    fov_x = original_spacing_x * ds.Columns
    fov_y = original_spacing_y * ds.Rows
    
    new_pixel_size_x = fov_x / target_resolution
    new_pixel_size_y = fov_y / target_resolution

    # 2. Cálculo de Z (Grosor/Espaciado de corte)
    # Intentamos el método de diferencia de posición entre cortes (más preciso)
    try:
        if len(slices) > 1:
            # Calculamos la diferencia absoluta en el eje Z entre el primer y segundo corte
            z_spacing = abs(slices[0].ImagePositionPatient[2] - slices[1].ImagePositionPatient[2])
        else:
            # Si solo hay un corte, no hay diferencia que calcular
            z_spacing = float(ds.SliceThickness)
            
    except (AttributeError, IndexError, TypeError):
        # Si fallan las coordenadas, intentamos leer el metadato SliceThickness
        try:
            z_spacing = float(ds.SliceThickness)
        except AttributeError:
            # Si nada de lo anterior existe, asignamos un valor por defecto (usualmente 1.0 o NaN)
            z_spacing = 1.0
            print("Advertencia: No se pudo determinar el espaciado Z. Usando valor por defecto 1.0")

    return [new_pixel_size_x, new_pixel_size_y, z_spacing]
"""
# --- Llamada a la función ---
#voxel_dims = get_voxel_size(dicom_slices, target_resolution=512)

image = nib.load("mascara_optimizada.nii.gz")
data = image.get_fdata()
header = image.header
voxel_dims = header.get_zooms()

print(f"Dimensiones del Voxel finales:")
print(f"X: {voxel_dims[0]:.4f} mm")
print(f"Y: {voxel_dims[1]:.4f} mm")
print(f"Z: {voxel_dims[2]:.4f} mm")

Dimensiones del Voxel finales:
X: 0.7812 mm
Y: 0.7812 mm
Z: 1.0000 mm


In [11]:
nombre_base = os.path.basename(mask_path)

coord_reales = [int(coord[0]*voxel_dims[0])/1000, int(coord[1]*voxel_dims[1])/1000, int(coord[2]*voxel_dims[2])/1000]
with open(f"{nombre_base}_coordenadas.txt", "w") as f:
    f.write(f"{coord_reales}\n")


**Obtener STL con tamaño real y suavizado**

In [12]:
import nibabel as nib
import pyvista as pv

def convert_nii_to_stl(nii_path, stl_path, voxel_dims, threshold=0.05):
    print(f"Cargando archivo: {nii_path}...")
    
    # 1. Cargar el archivo NIfTI
    try:
        img = nib.load(nii_path)
        data = img.get_fdata()
    except Exception as e:
        print(f"Error al cargar el archivo: {e}")
        return

    # 2. Configurar la rejilla 3D
    grid = pv.ImageData()
    grid.dimensions = data.shape
    
    # -------------------------------------------------------------------------
    # EL CAMBIO CLAVE: Asignar el tamaño de vóxel físico a la separación de la malla
    # Esto es lo que transforma los "píxeles" en milímetros reales.
    # -------------------------------------------------------------------------
    grid.spacing = voxel_dims 
    
    # Conservamos el origen espacial del NIfTI (por si necesitas alineación global)
    grid.origin = img.affine[:3, 3] 
    
    # Aplanar los datos para inyectarlos en la rejilla
    grid.point_data["values"] = data.flatten(order="F")

    # 3. Generar la superficie (Marching Cubes)
    print(f"Generando superficie con umbral {threshold}...")
    surface = grid.contour([threshold])

    # 4. Procesamiento final y Guardado
    if surface.n_points > 0:
        print("Suavizando la malla...")
        surface = surface.smooth(n_iter=750)
        
        surface.save(stl_path)
        print(f"¡Éxito! Archivo STL guardado a escala real en: {stl_path}")
    else:
        print("Error: No se encontró anatomía con ese umbral.")

# =========================================================================
# EJECUCIÓN
# =========================================================================
input_file = "mascara_optimizada.nii.gz"

# Tu variable con los tamaños de vóxel físicos (Ejemplo: [X, Y, Z] en mm)
dimensiones_reales_voxel = voxel_dims

# Ajuste del umbral
umbral_de_gris = 0.05

# Llamada a la función pasando la nueva variable
convert_nii_to_stl(
    nii_path=input_file, 
    stl_path=output_file, 
    voxel_dims=dimensiones_reales_voxel, 
    threshold=umbral_de_gris
)

Cargando archivo: mascara_optimizada.nii.gz...
Generando superficie con umbral 0.05...
Suavizando la malla...
¡Éxito! Archivo STL guardado a escala real en: volumen.stl
